In [ ]:
# Build a multi-step intelligent advisor system using agent workflows.

# 💼 Problem Statement
# Students need guidance on career paths based on their skills.

# Your system should:

# Analyze student profile
# Suggest career roles
# Generate learning roadmap
# 👥 Agents to Build
# Profile Analyzer Agent
# Understands student background
# Career Recommendation Agent
# Suggests suitable roles
# Skill Gap Agent
# Identifies missing skills
# Learning Path Agent
# Creates step-by-step roadmap

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain.chat_models import init_chat_model

In [ ]:
# =========================
# 1. LLM Setup
# =========================
llm = init_chat_model(model="mistral-medium")

In [ ]:
# =========================
# 2. State Definition
# =========================
class CareerState(TypedDict):
    profile: str
    level: str
    roles: str
    skill_gaps: str
    roadmap: str

In [ ]:
# =========================
# 3. Agents
# =========================

# 🔹 Profile Analyzer
def profile_analyzer(state: CareerState):
    profile = state["profile"]

    prompt = f"""
    Analyze the student profile and determine:
    1. Skill level (Beginner / Intermediate / Advanced)
    2. Key skills

    Profile: {profile}
    """

    result = llm.invoke(prompt).content

    # crude parsing (you can improve with JSON output)
    level = "Beginner"
    if "Intermediate" in result:
        level = "Intermediate"
    elif "Advanced" in result:
        level = "Advanced"

    return {"level": level}


# 🔹 Career Recommendation Agent
def career_agent(state: CareerState):
    profile = state["profile"]

    prompt = f"""
    Suggest 2-3 suitable career roles for this student.

    Profile: {profile}
    """

    roles = llm.invoke(prompt).content
    return {"roles": roles}


# 🔹 Skill Gap Agent
def skill_gap_agent(state: CareerState):
    roles = state["roles"]
    profile = state["profile"]

    prompt = f"""
    Based on the profile and roles, identify missing skills.

    Profile: {profile}
    Roles: {roles}
    """

    gaps = llm.invoke(prompt).content
    return {"skill_gaps": gaps}


# 🔹 Learning Path Agent
def learning_path_agent(state: CareerState):
    gaps = state["skill_gaps"]
    level = state["level"]

    prompt = f"""
    Create a step-by-step learning roadmap.

    Skill gaps: {gaps}
    Level: {level}

    If Beginner → start from fundamentals
    If Intermediate → focus on projects + advanced topics
    """

    roadmap = llm.invoke(prompt).content
    return {"roadmap": roadmap}

In [ ]:
# =========================
# 4. Conditional Logic
# =========================
def route_by_level(state: CareerState):
    if state["level"].lower() == "beginner":
        return "beginner_path"
    else:
        return "advanced_path"

In [ ]:
# =========================
# 5. Graph Construction
# =========================
builder = StateGraph(CareerState)

# Nodes
builder.add_node("profile", profile_analyzer)
builder.add_node("career", career_agent)
builder.add_node("skill_gap", skill_gap_agent)
builder.add_node("beginner_path", learning_path_agent)
builder.add_node("advanced_path", learning_path_agent)

# Flow
builder.add_edge(START, "profile")
builder.add_edge("profile", "career")
builder.add_edge("career", "skill_gap")

builder.add_conditional_edges(
    "skill_gap",
    route_by_level,
    {
        "beginner_path": "beginner_path",
        "advanced_path": "advanced_path",
    }
)

builder.add_edge("beginner_path", END)
builder.add_edge("advanced_path", END)

# Compile
graph = builder.compile()

In [ ]:
# =========================
# 6. Run Function
# =========================
def run_advisor(profile_text):
    result = graph.invoke({
        "profile": profile_text,
        "level": "",
        "roles": "",
        "skill_gaps": "",
        "roadmap": ""
    })
    return result

In [ ]:
# =========================
# 7. Test Cases
# =========================
profiles = [
    "I know Python basics and statistics, and I want to become a Data Scientist",
    "I have experience in Java, databases, and backend development"
]

for p in profiles:
    print("PROFILE:", p)
    output = run_advisor(p)

    print("\nLevel:", output["level"])
    print("\nRoles:", output["roles"])
    print("\nSkill Gaps:", output["skill_gaps"])
    print("\nRoadmap:", output["roadmap"])
    print("="*60)

C:\Users\sk165\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROFILE: I know Python basics and statistics, and I want to become a Data Scientist

Level: Intermediate

Roles: Given your background in **Python basics and statistics** and your goal to become a **Data Scientist**, here are **2-3 suitable career roles** to consider as stepping stones or entry points into the field:

### **1. Data Analyst**
   - **Why?** A natural starting point for aspiring Data Scientists, as it builds foundational skills in data cleaning, visualization, and basic modeling.
   - **Skills Used:** Python (Pandas, NumPy, Matplotlib/Seaborn), SQL, statistical analysis.
   - **Next Step:** Transition to more advanced roles (e.g., Data Scientist) by learning machine learning and deeper statistical modeling.

### **2. Business Intelligence (BI) Analyst**
   - **Why?** Focuses on data-driven decision-making, often using Python and statistical insights to generate reports and dashboards.
   - **Skills Used:** Python (data manipulation, basic stats), SQL, BI tools (Tableau, P